# Copyright 2022 Cognite AS
# (Advanced) Forecasting the thermal resistance 

Along with the Physics calculations, if we need to predict the values of thermal resistance for future timesteps, based on trend or other factors, we can quickly build a model which will forecast the thermal resistance, based on recent history and it can be used to monitor the performance of the heat exchanger and using predictions, we can detect any upcoming issues or problems in the system.

## Authentication

Create a client to communicate with the cognite data platform (CDF)

In [ ]:
from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import Token
from msal import PublicClientApplication


# Enter your name here
YOUR_NAME = "Andris-Piebalgs"

# Contact Project Administrator to get these
TENANT_ID="48d5043c-cf70-4c49-881c-c638f5796997"
CLIENT_ID="1c224c70-1df2-4f05-964a-753a6731748a"
CDF_CLUSTER="westeurope-1"
COGNITE_PROJECT="ds-cognitefunctions"

SCOPES = [f"https://{CDF_CLUSTER}.cognitedata.com/.default"]
TOKEN_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
AUTHORITY_HOST_URI = "https://login.microsoftonline.com"
AUTHORITY_URI = AUTHORITY_HOST_URI + "/" + TENANT_ID
PORT = 53000

def authenticate_azure():

    app = PublicClientApplication(client_id=CLIENT_ID, authority=AUTHORITY_URI)

    # interactive login - make sure you have http://localhost:port in Redirect URI in App Registration as type "Mobile and desktop applications"
    creds = app.acquire_token_interactive(scopes=SCOPES, port=PORT)
    return creds

creds = authenticate_azure()


cnf = ClientConfig(client_name="my-special-client", project=COGNITE_PROJECT, credentials=Token(creds["access_token"]), base_url=f"https://{CDF_CLUSTER}.cognitedata.com")
client = CogniteClient(cnf)

## Machine Learning to Forecast the thermal resistance

Along with the Physics calculations, if we need to predict the values of thermal resistance for future timesteps, based on trend or other factors, we can quickly build a model which will forecast the thermal resistance, based on recent history and it can be used to monitor the Performance of the Heat exchanger and using predictions, we can detect any upcoming issues or problems in the system.

First install the library we'll use for forecasting

In [ ]:
%pip install -q Prophet

Retrieve the data required for the Heat Exchanger

In [ ]:
column_names = ["T_cold_IN","T_cold_OUT","T_hot_IN","T_hot_OUT","Flow_cold","Flow_hot"]
ts_exids = ['pi:163657','pi:163658','pi:160887','pi:191092','pi:163374','pi:160184']

import datetime
from datetime import timedelta
start_date = datetime.datetime(2018, 8, 1)
end_date = start_date + timedelta(days=30)

df = client.datapoints.retrieve_dataframe(external_id=ts_exids,
                                                     aggregates=['average'],
                                                     granularity='1h',
                                                     start=start_date,
                                                     end=end_date,
                                                     include_aggregate_name=False
                                                     )
df.fillna(method="ffill", inplace=True)
df.columns = column_names
df.head()

In [ ]:
from tools import thermal_resistance

# Calculate the thermal resistance
df['TR'] = df.apply(lambda x:thermal_resistance(x),axis=1)

Now, let's format the data suitable for Prophet functions

In [ ]:
df2 = df.copy()[['TR']].reset_index()
df2 = df2.rename(columns={'index':"ds",'TR':"y"})

df2.head()

In [ ]:
df.shape

In [ ]:
from prophet import Prophet

In [ ]:
m = Prophet()
m.fit(df2)

### Forecast for next 15 days

Let's add 10% increase from the historical mean as the threshold for monitoring

In [ ]:
future = m.make_future_dataframe(periods=24*15, freq='H')
future['cap'] = 1.1*df['TR'].mean() #
fcst = m.predict(future)
fig = m.plot(fcst)

It shows that as per current trend based on last few weeks, data, it will cross the threshold very soon. This information can be utilized by the maintenance department or the operations.

Next, we can create the cognite function to run this model daily and save the forecasts in CDF for monitoring purposes.

## Running ML predictions with cognite functions

Add the below code the the handler.py file. Additionally, we need to create a requirements.txt file to add the Prophet library as dependency to be installed.

In [ ]:
from cognite.client.data_classes import TimeSeries
from math import log
import datetime
from datetime import timedelta
from prophet import Prophet

def thermal_resistance(x):
    '''Function to calculate Thermal Resistance'''
    # Some constants like Correction factor, Area and Cp values
    F = 0.8
    A = 1.0
    Cp_hot = 2.4
    # Calculate the cross temperature differences
    x['dT1'] = x['T_hot_IN'] - x['T_cold_OUT']
    x['dT2'] = x['T_hot_OUT'] - x['T_cold_IN']
    # Calculate the numerator and denominator for the thermal resistance calculation
    temp1 = A*F*(x['dT1']-x['dT2'])/log(x['dT1']/x['dT2'])
    temp2 = x['Flow_hot']*Cp_hot*(x['T_hot_IN']-x['T_hot_OUT'])
    tr = temp1/temp2
    return tr

def thermal_resistance_forecast(df):
    '''Function to forecast the Thermal Resistance'''
    df2 = df.copy()[['TR']].reset_index()
    df2 = df2.rename(columns={'index':"ds",'TR':"y"})
    m = Prophet()
    m.fit(df2)
    future = m.make_future_dataframe(periods=24*15, freq='H')
    future['cap'] = 1.1*df['TR'].mean() #
    fcst = m.predict(future)
    fcst_df = fcst[['ds','yhat']].set_index('ds')
    fcst_df.columns = ['TR']
    return fcst_df

def create_and_save_time_series_data(client,data,ts_external_id,data_set_id):
    '''Function to create the time series and save the data'''
    cdf_ts = client.time_series.retrieve(external_id=ts_external_id)
    if cdf_ts is None:
        ts = TimeSeries(external_id=ts_external_id,name=ts_external_id, unit = 'm2K/W',data_set_id=data_set_id)
        client.time_series.create(ts)
        print("Created time series")
    else:
        print("Existing Time Series")
        
    dps = []
    for index, r in data.iterrows():
        dps= dps+[{"timestamp": r.name, "value": r['TR']}]
        client.datapoints.insert(datapoints = dps,external_id = ts_external_id)


def handle(client,data=None, secrets=None, function_call_info=None):
    """Handler Function to be Run/Deployed
    Args:
        client : Cognite Client (not needed, it's availble to it, when deployed)
        data : data needed by function
        secrets : Any secrets it needs
        function_call_info : any other information about function

    Returns:
        response : response or result from the function 
    """
    data_set_id = client.data_sets.retrieve(external_id='functions-playground').id
    ts_exids = ['pi:163657','pi:163658','pi:160887','pi:191092','pi:163374','pi:160184']
    column_names = ["T_cold_IN","T_cold_OUT","T_hot_IN","T_hot_OUT","Flow_cold","Flow_hot"]
    your_name = data['your_name']
    # Retrieve the data
    start_date = datetime.datetime(2018, 8, 1)
    end_date = start_date + timedelta(days=30)
    df = client.datapoints.retrieve_dataframe(external_id=ts_exids,
                                                        aggregates=['average'],
                                                        granularity='1h',
                                                        start=start_date,
                                                        end=end_date,
                                                        include_aggregate_name=False
                                                        )
    df.fillna(method="ffill", inplace=True)
    df.columns = column_names
    # Calculate the Thermal resistance
    df['TR'] = df.apply(lambda x:thermal_resistance(x),axis=1)
    # Forecast the Thermal resistance
    fcst_df = thermal_resistance_forecast(df)
    # Save the Results as time series
    create_and_save_time_series_data(client,df[['TR']],f"hx_thermal_resistance_{your_name}", data_set_id=data_set_id)
    create_and_save_time_series_data(client,fcst_df[['TR']],f"hx_thermal_resistance_forecast_{your_name}", data_set_id=data_set_id)
    # Return the result as json
    result = df[['TR']].to_json()
    return result

We can create a new folder and save the updated handler.py & requirements.txt file there. There is already a folder in solutions if you would like to use that

In [ ]:
function_external_id = f"fthermal-perf-forecast-{YOUR_NAME}"

#folder_dir = <YOUR_CUSTOM_FOLDER> # location of your code and folder directory
folder_dir ='../solutions/hx-thermal-resistance-forecast/'


function = client.functions.create(
    name="Thermal Performance Forecasting Function",
    external_id=function_external_id,
    folder=folder_dir,
    description="Function to Calculate & Forecast Thermal Performance of Discharge Coolers",
    owner=YOUR_NAME
    )
# Note : if this cell is giving you errors :
# Just restart the notebook, create the cognite client and run this cell again.

In [ ]:
from function_tools import status_check

status_check(function)

In [ ]:
# Call your function

call = function.call(data={'your_name':YOUR_NAME})
call

In [ ]:
handle(client, {'your_name':YOUR_NAME})

In [ ]:
# Check if time series has been written back to and plot both the value and the forecast
import datetime as datetime

df=client.datapoints.retrieve(external_id = [f"hx_thermal_resistance_{YOUR_NAME}",f"hx_thermal_resistance_forecast_{YOUR_NAME}"], start=datetime.datetime(2018, 8, 1), end=datetime.datetime(2018, 10, 1)).to_pandas()
df.plot()

## Cleanup

You can delete the created/deployed function using Code or UI. This will also delete the schedules along with them.

In [ ]:
client.functions.delete(id=function.id)